# Using `torch-hungarian` 0.2 from RF-DETR

[RF-DETR](https://github.com/roboflow/rf-detr) 1.12.1 solves its matching problems with `torch_linear_assignment.batch_linear_assignment` and then converts the result to index pairs itself. Release 0.2.0 adds `batch_linear_assignment_indices`, which returns those pairs directly. This notebook shows what changes in RF-DETR's `_solve_to_indices`, proves the old and new code return the same matching, times both on RF-DETR-shaped problems, and ends with the A/B run that decides whether the new release is as fast as the shipped one.

**Runtime requirement:** a GPU with compute capability 8.0 or newer (L4, A100 or H100). A T4 (`sm_75`) only runs the SciPy fallback, so the notebook stops at the GPU gate instead of timing the wrong backend. **Runtime → Run all** runs it from top to bottom; expect a few minutes, most of it the install and the two benchmark runs in section 4.

| Section | Question it answers |
| --- | --- |
| 1 | What does RF-DETR's solve look like today, and what is the replacement? |
| 2 | Do the old and new code return identical `(rows, cols)`? |
| 3 | How long does each take at RF-DETR's batch sizes? |
| 4 | Is the new release as fast as the shipped one on RF-DETR's shapes (A/B on this GPU)? |
| 5 | What exactly should RF-DETR change? |

## 0. Setup

The release is not on PyPI yet, so the package comes from the `develop` branch. Triton ships with the CUDA PyTorch wheel that Colab provides.

In [1]:
# Release 0.2.0 is not published yet; the development branch has the new function.
!pip install -q "torch-hungarian @ git+https://github.com/Borda/torch-hungarian.git@develop"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import statistics

import pandas as pd
import torch
from IPython.display import display

The GPU gate comes first: on an unsupported GPU the package silently uses SciPy, and every timing below would measure the wrong thing.

In [3]:
assert torch.cuda.is_available(), "Choose a GPU runtime: Runtime -> Change runtime type"
capability = torch.cuda.get_device_capability()
assert capability >= (8, 0), (
    f"{torch.cuda.get_device_name()} is sm_{capability[0]}{capability[1]}; the Triton backend needs sm_80 or newer"
)
print(torch.cuda.get_device_name(), f"sm_{capability[0]}{capability[1]}", "torch", torch.__version__)

NVIDIA L4 sm_89 torch 2.14.1+cu130


In [4]:
# Importing the new name fails fast if the install resolved to a release before 0.2.0.
from torch_linear_assignment import batch_linear_assignment, batch_linear_assignment_indices

## 1. RF-DETR's solve today and its replacement

RF-DETR folds every `(layer, image, group)` problem of a training step into one batch of shape `[problems, group_width, targets]`: 300 queries per group, targets padded to the largest count in the batch. Below is that batch for 16 images with 13 query groups, so 208 problems, padded to 30 targets. The costs are random, which is enough to compare two code paths on the same input.

In [5]:
GROUP_WIDTH, MAX_TARGETS, IMAGES, GROUPS = 300, 30, 16, 13
generator = torch.Generator().manual_seed(0)
stacked = torch.rand((IMAGES * GROUPS, GROUP_WIDTH, MAX_TARGETS), generator=generator).cuda()
print(tuple(stacked.shape), stacked.dtype, stacked.device)

(208, 300, 30) torch.float32 cuda:0


**Before.** This is RF-DETR 1.12.1's `rfdetr/models/_assignment.py::_solve_to_indices`. The solver returns one target per query (`-1` when a query is unmatched), so RF-DETR sorts a matched mask to list the matched queries in ascending order, gathers their targets, and copies both tensors to the host.

In [6]:
def solve_to_indices_before(stacked: torch.Tensor, num_matches: int) -> tuple[torch.Tensor, torch.Tensor]:
    assignment = batch_linear_assignment(stacked)
    matched = assignment >= 0
    # Stable and descending puts every matched query first, in ascending row order.
    order = torch.argsort(matched.to(torch.int8), dim=1, descending=True, stable=True)
    rows = order[:, :num_matches]
    cols = assignment.gather(1, rows)
    pair = torch.stack((rows, cols)).cpu()
    return pair[0], pair[1]

**After.** `batch_linear_assignment_indices` returns the same pairs directly: `rows` sorted within each problem and `cols[b, i]` the target of query `rows[b, i]`, both `torch.long` on the input's device with shape `(problems, min(width, targets))`. The mask, the sort and the gather disappear, and `num_matches` is no longer needed because the shape already carries it.

In [7]:
def solve_to_indices_after(stacked: torch.Tensor, num_matches: int) -> tuple[torch.Tensor, torch.Tensor]:
    rows, cols = batch_linear_assignment_indices(stacked)
    assert rows.shape[1] == num_matches
    pair = torch.stack((rows, cols)).cpu()
    return pair[0], pair[1]

## 2. The two return the same matching

Both functions run on the same batch. The shape is checked first because a wrong number of matches would otherwise surface as a confusing mismatch below.

In [8]:
NUM_MATCHES = min(GROUP_WIDTH, MAX_TARGETS)
rows_before, cols_before = solve_to_indices_before(stacked, NUM_MATCHES)
rows_after, cols_after = solve_to_indices_after(stacked, NUM_MATCHES)
assert rows_before.shape == rows_after.shape == (IMAGES * GROUPS, NUM_MATCHES)

In [9]:
# Exact equality, not closeness: both paths read the same solver output, and RF-DETR's parity tests require identical index pairs on continuous costs.
assert torch.equal(rows_before, rows_after)
assert torch.equal(cols_before, cols_after)
print("identical (rows, cols) for", rows_after.shape[0], "problems")

identical (rows, cols) for 208 problems


The new function also accepts `validate="off"`, which skips the two host reads the default makes (a NaN and `-inf` check before the launch, an infeasibility check after it) and allows CUDA graph capture after a warm-up. It is only safe when the costs are known to be finite and feasible: invalid costs then return unspecified indices instead of raising. The next cell checks that on valid input both modes agree.

In [10]:
rows_off, cols_off = batch_linear_assignment_indices(stacked, validate="off")
assert torch.equal(rows_off.cpu(), rows_after)
assert torch.equal(cols_off.cpu(), cols_after)

## 3. Timing at RF-DETR's batch sizes

A training step solves 52, 208 or 624 problems depending on batch size and decoder layers. Each call is timed after a warm-up with a device synchronize around it, so the number includes the host copy RF-DETR does, which is the cost a training step actually pays.

In [11]:
def median_ms(function, stacked, num_matches, repeats=30, warmup=5):
    for _ in range(warmup):
        function(stacked, num_matches)
    samples = []
    for _ in range(repeats):
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        function(stacked, num_matches)
        end.record()
        torch.cuda.synchronize()
        samples.append(start.elapsed_time(end))
    return statistics.median(samples)

In [12]:
rows = []
for problems in (52, 208, 624):
    batch = torch.rand((problems, GROUP_WIDTH, MAX_TARGETS), generator=generator).cuda()
    before = median_ms(solve_to_indices_before, batch, NUM_MATCHES)
    after = median_ms(solve_to_indices_after, batch, NUM_MATCHES)
    rows.append(
        {
            "problems": problems,
            "before_ms": round(before, 2),
            "after_ms": round(after, 2),
            "after / before": round(after / before, 2),
        }
    )
display(pd.DataFrame(rows))

,problems,before_ms,after_ms,after / before
0,52,0.80,0.73,0.92
1,208,0.82,0.76,0.92
2,624,1.02,0.98,0.95


Read the last column as a ratio of times: below `1.0` means the new code is faster. The two paths share the solver, so the gain is limited to the conversion RF-DETR no longer does. Treat a ratio near `1.0` as expected, not as a failure.

## 4. Is 0.2.0 as fast as the shipped kernel? (A/B on this GPU)

The release changed the kernel launch and the validation path, so the claim to check is that RF-DETR's shapes did not get slower. The cells below clone the repository, run the repository's own `rfdetr` benchmark preset at the shipped revision `881e5c3` and at `develop`, on this same GPU in this same session, and compare warm medians per shape. `--process-rounds 3` runs each case in three fresh processes, the default of the Modal lane.

In [13]:
import subprocess
from pathlib import Path

REPO = Path("/content/torch-hungarian")
BASELINE_REF = "881e5c3"
CANDIDATE_REF = "origin/develop"

if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/Borda/torch-hungarian.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin"], check=True)

CompletedProcess(args=['git', '-C', '/content/torch-hungarian', 'fetch', '-q', 'origin'], returncode=0)

In [14]:
import os


def run_rfdetr_preset(ref: str) -> Path:
    """Benchmark the `rfdetr` preset at one revision, importing the package from that checkout."""
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "--detach", ref], check=True)
    output = Path(f"/content/benchmark-{ref.replace('/', '-')}.jsonl")
    output.unlink(missing_ok=True)
    command = [
        "python",
        str(REPO / "tests/benchmark.py"),
        "--preset",
        "rfdetr",
        "--backends",
        "scipy,public_cuda,triton",
        "--dtypes",
        "float32",
        "--process-rounds",
        "3",
        "--output",
        str(output),
    ]
    # PYTHONPATH puts the checked-out sources ahead of the package installed in section 0, so each run measures its own revision.
    environment = {**os.environ, "PYTHONPATH": str(REPO / "src")}
    subprocess.run(command, cwd=REPO, env=environment, check=True)
    return output

In [15]:
baseline_records = run_rfdetr_preset(BASELINE_REF)

In [16]:
candidate_records = run_rfdetr_preset(CANDIDATE_REF)

The comparison uses the public call (`public_cuda`) with default validation, the entry point RF-DETR reaches, and reads the median across process rounds from each record. A ratio above `1.05` is flagged: it is the repository's tolerance for the RF-DETR shapes.

In [17]:
import json


def warm_medians(path: Path) -> dict[tuple[int, int, int], float]:
    records = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    medians = {}
    for record in records:
        if record["backend"] == "public_cuda" and record["status"] == "measured":
            shape = record["shape"]
            medians[(shape["batch"], shape["workers"], shape["tasks"])] = record["process_statistics"][
                "warm_median_ms"
            ]["median"]
    return medians

In [18]:
baseline, candidate = warm_medians(baseline_records), warm_medians(candidate_records)
assert baseline.keys() == candidate.keys() and baseline, "the two revisions must measure the same shapes"
table = pd.DataFrame(
    [
        {
            "batch": key[0],
            "tasks": key[2],
            "shipped_ms": round(baseline[key], 2),
            "develop_ms": round(candidate[key], 2),
            "develop / shipped": round(candidate[key] / baseline[key], 2),
        }
        for key in sorted(baseline)
    ]
)
table["slower than 1.05x"] = table["develop / shipped"] > 1.05
display(table)

,batch,tasks,shipped_ms,develop_ms,develop / shipped,slower than 1.05x
0,52,1,0.42,0.46,1.09,True
1,52,2,0.46,0.48,1.05,False
2,52,5,0.47,0.48,1.02,False
3,52,10,0.48,0.50,1.04,False
4,52,20,0.50,0.56,1.11,True
5,52,30,0.53,0.61,1.14,True
6,208,1,0.43,0.47,1.09,True
7,208,2,0.48,0.49,1.03,False
8,208,5,0.50,0.52,1.04,False
9,208,10,0.49,0.58,1.19,True


A row marked `True` means `develop` is more than 5% slower than the shipped kernel for that RF-DETR shape; no row marked means the claim holds on this GPU. One session on one GPU is evidence, not a record: the Modal benchmark workflow keeps the source-bound JSONL for the numbers that go in the README.

## 5. What RF-DETR should change

1. In `rfdetr/models/_assignment.py`, replace the body of `_solve_to_indices` with the "after" version from section 1: call `batch_linear_assignment_indices(stacked)`, stack `rows` and `cols`, and copy them to the host once.
2. In RF-DETR's `train` extra, relax the exact pin `torch-hungarian==0.1.0rc0` to `torch-hungarian>=0.2.0`. Until RF-DETR does, its users get none of the kernel or validation changes.
3. Keep the default `validate="full"` unless RF-DETR's matcher can guarantee finite costs; `validate="off"` is what makes the call capturable in a CUDA graph.